# Part 12 · Notebook 06 — Anomaly detection, the controller and reports

**Sessions:** S11–S12 (Anomaly detection & the controller · Reports & dashboards) · [Lesson plan](../../docs/lessons/PART_12_TRADING_PLATFORM_CAPSTONE.md) · graded labs in [`labs/part12/`](../../labs/part12/)

**You will:**
1. Build a streaming robust z-score, and see why latency is scored on the log scale.
2. Catch a slow drift that no single observation reveals, with CUSUM.
3. Replay a trading day with injected faults: detect each one, book fills idempotently, let the controller act.
4. Write the daily report, and judge a live record against the backtest's expected band.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
Everything runs offline on synthetic data with known answers (planted violations, injected faults, a strategy that stops working): no broker, Docker or network is needed.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p12lib.py is in notebooks/part12/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p12lib as p

p.use_course_style()

## 1. A streaming robust z-score

Mean and standard deviation are dragged around by the very outliers you are hunting. The **median** and the **MAD** (median absolute deviation, × 1.4826 so it matches σ for normal data) are not. The score of a new value uses only the values before it (the last `window`), and nothing is flagged until `min_obs` values have been seen.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
class MyRobustZ:
    def __init__(self, window=500, threshold=6.0, min_obs=50):
        self.buf, self.window, self.threshold, self.min_obs = [], window, threshold, min_obs

    def update(self, x):
        z = 0.0
        if len(self.buf) >= self.min_obs:                            # only the values BEFORE x, and only once warm
            arr = np.asarray(self.buf)
            med = ...                                                # ✍️ the median of the window
            mad = ...                                                # ✍️ the MAD × 1.4826, plus 1e-12 against a zero spread
            z = (x - med) / mad
        self.buf.append(x)
        if len(self.buf) > self.window:
            self.buf.pop(0)
        return z, abs(z) > self.threshold

def run(detector, xs):
    out = [detector.update(float(x)) for x in xs]
    return np.array([z for z, _ in out]), [i for i, (_, flag) in enumerate(out) if flag]

latencies = p.latency_samples()                                      # 300 order-ack latencies, one 400 ms spike at 200
z_raw, flags = p.check("RobustZ", p.attempt(run, MyRobustZ(), latencies), run(p.RobustZ(), latencies))
print(f"flagged: {flags}; z of the spike: {z_raw[200]:.1f}")

One flag, on the spike. Latencies are right-skewed, though, so the normal tail itself scores high on the raw scale. Compare the largest score on 20 clean days (no spike) with the log scale:

In [ ]:
clean = pd.DataFrame([{"raw": np.abs(run(p.RobustZ(), x)[0]).max(), "log": np.abs(run(p.RobustZ(), np.log(x))[0]).max()}
                      for x in (p.latency_samples(spike_at=None, seed=s) for s in range(1, 21))])
z_log, flags_log = run(p.RobustZ(), np.log(latencies))
print(f"largest |z| on 20 clean days: raw up to {clean['raw'].max():.1f}, log up to {clean['log'].max():.1f} (the alarm is at 6)")
print(f"the spike: z = {z_raw[200]:.1f} raw, {z_log[200]:.1f} log; flagged on the log scale: {flags_log}")
print("fat-tail check, a −$12,000 day when the risk model says σ = $2,000:", p.fat_tail_error(-12_000, 2_000))

On the raw scale an ordinary day already comes within a whisker of the alarm; on the log scale the clean days stay far below it and the spike still stands out. Score latency on the log scale. The fat-tail check is the P&L version of the same idea: a 6σ loss is more often a bad price, a missing position or a duplicated fill than a real market move, so the controller pauses everything and reconciles before anyone trusts the number.

## 2. Slow drift: CUSUM

A strategy's slippage creeping up by 0.8σ never produces one extreme value. CUSUM adds up the small excesses (`s⁺ = max(0, s⁺ + z − k)`, and the mirror image for drops) and alarms when the sum passes `h`, then starts again:

In [ ]:
rng = np.random.default_rng(1)
x = rng.normal(0, 1, 400)
x[200:] += 0.8                                                       # from t = 200 the mean is 0.8 σ higher
cusum, rz = p.Cusum(mean=0, sd=1), p.RobustZ()
alarms = [t for t, v in enumerate(x) if cusum.update(v)]
flags = [t for t, v in enumerate(x) if rz.update(v)[1]]
print("RobustZ flags:", flags, "| CUSUM alarms:", alarms)

fig, ax = plt.subplots()
ax.plot(x, lw=0.7, color="#8a8984", label="observations")
ax.plot(pd.Series(x).rolling(20).mean().to_numpy(), label="20-point mean")
for t in alarms:
    ax.axvline(t, color=p.PALETTE[7], lw=0.8)
ax.axvline(200, color="k", ls="--", lw=1, label="the drift starts")
ax.set(title="CUSUM alarms (red lines) on a 0.8σ drift", xlabel="observation")
ax.legend(loc="upper left")
plt.show()

The z-score never fires; CUSUM fires a few observations after the drift starts and keeps firing while it lasts. Each detector has a job: z-scores for spikes, CUSUM for drift.

## 3. The monitoring drill

`p.trading_day()` replays a session with five injected faults. `p.FAULTS` is the answer key. Each detector looks for one kind of fault:

In [ ]:
day = p.trading_day()
print("injected (the answer key):", p.FAULTS)
dets = ([("bad_tick", t) for t in p.bad_ticks(day["ticks"])] + [("stale_data", t) for t in p.stale_gaps(day["ticks"])]
        + [("latency_spike", i) for i in p.stream_flags(np.log(day["acks"]))]
        + [("runaway_order_rate", i) for i in p.stream_flags(day["order_counts"])])
print("detected:", dets)

naive = {}
for f in day["fills"]:                                               # book every fill message as it arrives
    naive[f["symbol"]] = naive.get(f["symbol"], 0) + f["qty"]
print("naive booking vs the broker:", p.reconcile_positions(naive, day["broker_positions"]))

Four faults caught where they were injected (the stale feed is reported at its last tick, 11,999, just before the 30-second silence). The fifth hides in the fills: after a reconnect the broker re-sent one fill, and booking it twice leaves our position different from the broker's. Book **idempotently**: each `fill_id` once, and report the ids that arrived more than once.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def book_fills(fills):
    seen, dups, pos = set(), [], {}
    for f in fills:
        ...                                                          # ✍️ a fill_id seen before: record it and skip it
        seen.add(f["fill_id"])
        pos[f["symbol"]] = pos.get(f["symbol"], 0) + f["qty"]
    return {k: v for k, v in pos.items() if v != 0}, dups

booked, dups = p.check("book_fills", p.attempt(book_fills, day["fills"]), p.book_fills(day["fills"]))
print("delivered twice:", dups, "| breaks after idempotent booking:", p.reconcile_positions(booked, day["broker_positions"]))

Now the **controller**: a policy table maps each anomaly to an action (alert, pause the strategy, pause everything and reconcile, trip the kill switch), and every decision is audited. The drill adds two anomalies to the lesson-plan policy:

In [ ]:
POLICY = {**p.CONTROLLER_POLICY, "bad_tick": "pause_strategy", "duplicate_fill": "pause_all_and_reconcile"}
audit = p.AuditLog()
approvals = p.ApprovalQueue(audit)
ctl = p.Controller(["mom", "pairs", "options"], audit, approvals, POLICY)
for anomaly, where in dets + [("duplicate_fill", d) for d in dups]:
    print(f"{anomaly:>20} at {str(where):>6} → {ctl.handle(anomaly, 'mom')}")
print("\nkilled:", ctl.killed, "| paused:", sorted(ctl.paused), "| needs reconcile:", ctl.needs_reconcile,
      "| audit chain intact:", audit.verify() == -1)
aid = approvals.request("resume", {"strategy": "pairs"}, requester="alice", reason="false alarm")
approvals.decide(aid, approver="bob", approve=True, reason_code="checked")
print("resume pairs with an approved request:", ctl.resume("pairs", aid), "(nothing resumes while the kill switch is tripped)")

quiet = p.trading_day(faults=False)
print("a clean day:", p.bad_ticks(quiet["ticks"]) + p.stale_gaps(quiet["ticks"]) + p.stream_flags(np.log(quiet["acks"]))
      + p.stream_flags(quiet["order_counts"]), "detections,", p.book_fills(quiet["fills"])[1], "duplicates")

Every fault detected, handled and audited, and a clean day raises nothing: a monitor that cries wolf gets muted, which is worse than no monitor. (The homework tunes thresholds on four weeks of paper data to stay under one false alarm a day.)

## 4. Reports and the backtest band

The **daily report** (sent by n8n after the close) covers P&L, return, drawdown, the number of fills and slippage against the arrival mid:

In [ ]:
rng = np.random.default_rng(0)
fills = pd.DataFrame({"side": rng.choice([1, -1], 12), "qty": rng.integers(1, 6, 12) * 100,
                      "arrival_mid": 100 + rng.normal(0, 1, 12).cumsum()})
fills["price"] = fills["arrival_mid"] * (1 + fills["side"] * rng.normal(3, 2, 12) / 1e4)   # about 3 bps paid on each fill
equity = pd.Series([100_000, 101_200, 100_700, 99_900, 100_450.0])
{k: round(v, 4) for k, v in p.daily_report(fills, equity).items()}

The monthly question is harder: *is the live record what the backtest promised?* Live results are short and noisy, so compare them with a **band**: resample the backtest's daily returns into many 20-day paths (with replacement), add each path up, and take the 5% and 95% quantiles for every day.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def backtest_band(daily_returns, horizon=20, n_boot=2000, q=(0.05, 0.95), seed=0):
    r = np.asarray(daily_returns, dtype=float)
    rng = np.random.default_rng(seed)
    paths = ...                                                      # ✍️ (n_boot, horizon) resampled returns, summed along each path
    lo, hi = np.quantile(paths, q, axis=0)
    return pd.DataFrame({"lower": lo, "upper": hi}, index=range(1, horizon + 1))

bt, live = p.track_record()                                          # 1,000 backtest days and 20 live days per strategy
band = p.check("backtest_band", p.attempt(backtest_band, bt["options"]), p.backtest_band(bt["options"]))
band.iloc[[0, 4, 9, 19]]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharey=True)
for ax, s in zip(axes, live.columns):
    b = p.backtest_band(bt[s], horizon=len(live))
    ax.fill_between(b.index, b["lower"], b["upper"], alpha=0.25, label="backtest 5–95%")
    ax.plot(b.index, live[s].cumsum().to_numpy(), marker="o", ms=3, label="live")
    ax.set(title=f"{s}: inside on {p.inside_band(live[s], b).mean():.0%} of days", xlabel="live day")
axes[0].set_ylabel("cumulative return")
axes[0].legend(loc="lower left")
plt.show()
p.attribution(live * 1_000_000).round(2)                            # daily P&L on $1m per strategy

Momentum stays inside its band. Pairs runs *above* its band for a while: better than promised, which calls for a look (a data or accounting error can flatter too) rather than a demotion. The options strategy falls below its band within a week and keeps falling: something changed (the synthetic record has it bleeding 0.8% a day). Attribution needs the same care: with a losing total, the shares flip sign (options is almost twice the net loss, and the other two offset part of it), so read the totals first. A band turns "it's just a bad week" into a testable statement, and notebook 10 turns it into weekly scale / hold / demote decisions.

## Wrap-up

* Robust z-scores for spikes (latency on the log scale), CUSUM for drift, a fat-tail check on P&L.
* Book fills idempotently; the controller maps anomalies to audited actions; resuming needs an approval.
* A clean day must stay quiet.
* Judge live results against a band built from the backtest, not against a single number.
* Graded versions: `labs/part12/week43_monitoring` (`RobustZ`, `fat_tail_error`, `Cusum`, `Controller`, `daily_report`, `backtest_band`, `inside_band`, `attribution`) and the drill in `labs/part12/clinic_w3_monitoring_drill`.